# PearlRoute Vision

PearlNet, a custom CNN trained from scratch. All logic is imported from the `pearlroute` package.

## 1. Setup
We import the tools we need and load the three config files (data, model, training).
The real code lives in the `pearlroute` package, so this notebook stays short and just *uses* it.
`SEED` makes random choices (like which sample images we show) the same every run.

In [ ]:
import json
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # hide TensorFlow's low-level info/warning messages

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from pearlroute.config import load_data_config, load_model_config, load_train_config
from pearlroute.data.loaders import list_images, load_class_weights, load_datasets
from pearlroute.preprocessing.stats import compute_channel_stats, load_stats, save_stats
from pearlroute.preprocessing.transforms import read_and_resize

data_cfg, model_cfg, train_cfg = load_data_config(), load_model_config(), load_train_config()
CLASSES = data_cfg.class_names
SEED = data_cfg.seed

# Chart style: light background, quiet grid, no top/right borders.
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": "#52514e",
    "axes.labelcolor": "#0b0b0b", "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
    "grid.color": "#e4e3dd", "grid.linewidth": 0.8, "axes.axisbelow": True, "font.size": 10,
})
SPLIT_COLOURS = {"train": "#2a78d6", "val": "#eb6834", "test": "#1baf7a"}
print("classes:", CLASSES)
print("image size:", model_cfg.image_size, "| batch size:", train_cfg.batch_size)

## 2. How many images do we have?
`counts.json` is written by the cleaning and splitting scripts. **raw** is what we downloaded,
**clean** is what survived the quality checks (duplicates, black-and-white photos, tiny images).
The split table shows how the clean images were shared out; demo and retrain images are kept aside.

In [ ]:
counts = json.loads(data_cfg.paths.counts_json.read_text())
cleaning = pd.DataFrame({"raw": counts["cleaning"]["raw"], "clean": counts["cleaning"]["clean"]})
cleaning["dropped"] = cleaning["raw"] - cleaning["clean"]
display(cleaning)

split_table = pd.DataFrame(counts["split"]["per_class"]).T[["train", "val", "test", "retrain", "demo"]]
display(split_table)
print("dropped by reason:", counts["cleaning"]["dropped_by_reason"])

## 3. Chart: images per class in each split
Each bar is one class, cut into train (blue), val (orange) and test (aqua); the number is the total.
All classes should be roughly the same length, and every bar should be about 70 / 15 / 15.
If one class were much shorter, the model would see it less and probably predict it worse.

In [ ]:
order = split_table.sort_values("train").index          # shortest at the bottom
fig, ax = plt.subplots(figsize=(8, 3.6))
left = np.zeros(len(order))
for split, colour in SPLIT_COLOURS.items():
    values = split_table.loc[order, split].to_numpy()
    ax.barh(order, values, left=left, color=colour, height=0.6, label=split,
            edgecolor="#fcfcfb", linewidth=2)               # thin white gap between segments
    left += values
for y, total in enumerate(left):
    ax.text(total + 8, y, f"{int(total)}", va="center", color="#0b0b0b", fontsize=9)
ax.set_xlabel("images")
ax.set_title("Images per class: train / val / test", loc="left", color="#0b0b0b")
ax.grid(axis="y", visible=False)
ax.legend(ncol=3, frameon=False, loc="lower right", bbox_to_anchor=(1, 1.0))
ax.set_xlim(0, left.max() * 1.1)
plt.tight_layout()
plt.show()

## 4. No photographer appears in two splits
One photographer often takes many similar photos of the same place. If some went to train and
some to test, the test would be too easy (the model has already "seen" that scene).
So the split keeps each photographer in one split only; this cell checks that it worked.

In [ ]:
splits = pd.read_csv(data_cfg.paths.splits_csv, dtype=str, keep_default_na=False)
in_splits = splits[splits["split"].isin(["train", "val", "test"])]
leaks = (in_splits.groupby("group")["split"].nunique() > 1).sum()
print("photographers in more than one split:", leaks)
print("photographers per split:", in_splits.groupby("split")["group"].nunique().to_dict())
assert leaks == 0

## 5. Load the images with tf.data
`load_datasets` reads every image, resizes it to a square, and groups the images into batches.
Only the training set is shuffled, so the model doesn't see the classes in a fixed order.
A batch is a 4-D array: (images in batch, height, width, 3 colour channels). Labels are class numbers.

In [ ]:
raw_sets = load_datasets(data_cfg, model_cfg, train_cfg)          # pixels still 0-255
images, labels = next(iter(raw_sets["train"]))
print("batch shape:", images.shape, "| dtype:", images.dtype)
print("pixel range:", float(tf.reduce_min(images)), "to", float(tf.reduce_max(images)))
print("first labels:", labels[:8].numpy(), "->", [CLASSES[i] for i in labels[:4].numpy()])
print("class weights:", load_class_weights(data_cfg), "(None = classes are balanced enough)")

## 6. What the model actually sees
These are random training images after resizing to the model's input size, one row per class.
Photos are squashed to a square rather than cropped, so nothing is cut off; a little stretching is normal.
This is a quick sanity check that every class folder holds the kind of photo its name promises.

In [ ]:
paths, path_labels = list_images(data_cfg.paths.processed_dir / "train", CLASSES)
paths, path_labels = np.array(paths), np.array(path_labels)
rng = np.random.default_rng(SEED)
per_row = 6
fig, axes = plt.subplots(len(CLASSES), per_row, figsize=(per_row * 1.6, len(CLASSES) * 1.75))
for row, name in enumerate(CLASSES):
    picks = rng.choice(paths[path_labels == row], per_row, replace=False)
    for col, path in enumerate(picks):
        ax = axes[row, col]
        ax.imshow(read_and_resize(tf.constant(path), model_cfg.image_size).numpy())
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        for spine in ax.spines.values():
            spine.set_visible(False)
    axes[row, 0].set_ylabel(name.replace("_", "\n"), rotation=0, ha="right", va="center", fontsize=9)
fig.suptitle(f"Random training images at {model_cfg.image_size} x {model_cfg.image_size}", x=0.02, ha="left")
plt.tight_layout()
plt.show()

## 7. Normalisation: work out the average colour of the training set
Normalising means: for each colour channel, subtract the average and divide by the spread (std).
Afterwards pixel values sit around 0 instead of 0-255, which helps the network train faster and more stably.
We use **training images only**: peeking at val/test would leak information about them.
The numbers are saved to `data/stats.json` so the app normalises uploaded photos exactly the same way.

In [ ]:
mean, std, n = compute_channel_stats(raw_sets["train"])
save_stats(data_cfg.paths.stats_json, mean, std, model_cfg.image_size, n)
print(f"computed on {n} training images")
print("mean (R, G, B):", mean)
print("std  (R, G, B):", std)

## 8. Chart: pixel values before and after normalisation
Left: how often each pixel value (0-255) appears in one batch, for red, green and blue.
Right: the same pixels after normalising, now centred on 0 with most values between about -2 and 2.
The shape doesn't change, only the scale moves; that's exactly what normalising should do.

In [ ]:
stats = load_stats(data_cfg.paths.stats_json)
norm_sets = load_datasets(data_cfg, model_cfg, train_cfg, stats=stats)
before, _ = next(iter(raw_sets["val"]))
after, _ = next(iter(norm_sets["val"]))            # same images: val is not shuffled

channels = [("R", "#e34948", "-"), ("G", "#008300", "--"), ("B", "#2a78d6", ":")]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.4))
for ax, batch, bins, title in [(ax1, before, np.linspace(0, 255, 52), "Before: pixel values 0-255"),
                               (ax2, after, np.linspace(-3, 3, 52), "After: normalised")]:
    for c, (name, colour, style) in enumerate(channels):
        values = batch.numpy()[..., c].ravel()
        hist, edges = np.histogram(values, bins=bins, density=True)
        centres = (edges[:-1] + edges[1:]) / 2
        ax.plot(centres, hist, color=colour, linestyle=style, linewidth=2)
        peak = hist.argmax()
        ax.annotate(name, (centres[peak], hist[peak]), xytext=(4, 4), textcoords="offset points",
                    color="#0b0b0b", fontsize=9, fontweight="bold")
    ax.set_title(title, loc="left", color="#0b0b0b")
    ax.set_xlabel("pixel value")
    ax.set_yticks([])
ax1.set_ylabel("how common")
ax2.axvline(0, color="#52514e", linewidth=1)
fig.legend([plt.Line2D([], [], color=c, linestyle=s, linewidth=2) for _, c, s in channels],
           [n for n, _, _ in channels], ncol=3, frameon=False, loc="upper right")
plt.tight_layout()
plt.show()

## 9. Check: are the normalised values centred?
On the training set the mean should be almost exactly 0 and the std almost exactly 1, by construction.
Validation images were not used to compute the stats, so they land close to 0 and 1 but not exactly.
Close is good: it means val photos look like train photos, colour-wise.

In [ ]:
def overall_mean_std(dataset):
    m, s, _ = compute_channel_stats(dataset)
    return np.round(m, 3), np.round(s, 3)

for split in ["train", "val"]:
    m, s = overall_mean_std(norm_sets[split])
    print(f"{split:5s} mean {m}  std {s}")

## 10. Augmentation: small random changes to training photos
Each epoch, every training photo gets randomly flipped left-right, tilted a little, zoomed, shifted,
and made brighter/darker. The model never sees the exact same picture twice, so it has to learn
the subject (a raft, a gorilla) instead of memorising pixels. We skip upside-down flips and colour
shifts, because skies are up and colour matters here (green forest, white water).

In [ ]:
from pearlroute.preprocessing.augmentation import make_augment

print("settings:", train_cfg.augmentation.model_dump())
augment = make_augment(train_cfg.augmentation, seed=SEED)

## 11. Chart: one photo, many versions
The left column is the original photo; the other columns are what augmentation might produce from it.
Every version should still clearly show the same experience. If a change made the photo
unrecognisable (e.g. the raft cut off), the settings in `configs/train.yaml` would be too strong.

In [ ]:
# Pick one training photo per class whose title names the subject, so the example is a clear one.
train_rows = splits[splits["split"] == "train"]
rng = np.random.default_rng(SEED)
examples = []
for name, word in [("whitewater_rafting", "raft"), ("gorilla_primate_trekking", "gorilla"), ("waterfalls", "falls")]:
    rows = train_rows[(train_rows["label"] == name) & train_rows["title"].str.contains(word, case=False)]
    examples.append(str(data_cfg.paths.raw_dir / rng.choice(rows["filename"].to_numpy())))
versions = 6
fig, axes = plt.subplots(len(examples), versions, figsize=(versions * 1.7, len(examples) * 1.8))
for row, path in enumerate(examples):
    original = tf.cast(read_and_resize(tf.constant(path), model_cfg.image_size), tf.float32)
    batch = tf.repeat(original[None], versions - 1, axis=0)        # 5 copies of the same photo
    shown = [original] + list(augment(batch))
    for col, img in enumerate(shown):
        ax = axes[row, col]
        ax.imshow(np.asarray(img).astype("uint8"))
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        for spine in ax.spines.values():
            spine.set_visible(False)
    axes[row, 0].set_title("original", fontsize=9, loc="left")
    axes[row, 1].set_title("augmented ->", fontsize=9, loc="left")
fig.suptitle("Augmentation: same photo, different random changes", x=0.02, ha="left")
plt.tight_layout()
plt.show()

## 12. The full training pipeline
This is exactly what the model will be fed: resize -> augment (train only) -> normalise.
Augmentation runs while the images are loaded, so it costs a little time each epoch but no disk space.
Validation and test images are never augmented: we want to score the model on real, unchanged photos.

In [ ]:
import time

train_sets = load_datasets(data_cfg, model_cfg, train_cfg, augment=augment, stats=stats)
start = time.time()
batches = sum(1 for _ in train_sets["train"])
print(f"one training epoch of images: {batches} batches in {time.time() - start:.1f}s")
x, y = next(iter(train_sets["train"]))
print("train batch:", x.shape, "| mean", round(float(tf.reduce_mean(x)), 2), "| std", round(float(tf.math.reduce_std(x)), 2))
x_val, _ = next(iter(train_sets["val"]))
x_val_plain, _ = next(iter(norm_sets["val"]))
print("val batch unchanged by augmentation:", bool(np.allclose(x_val.numpy(), x_val_plain.numpy())))

## Next steps
- Model A: a simple CNN baseline. Model B: PearlNet.
- Train both on `train_sets`, compare them on the test set, and save the better model with `data/stats.json`.